# 13. Final Benchmark Tables: High, Intermittent, Low

This notebook rebuilds the three comparative tables directly from the original model benchmark notebooks:

- `04_walmart_repo_lstm_impl2_benchmark_products.ipynb`
- `05_lstm_deep_learning_time_series_forecasting_benchmark_products.ipynb`
- `06_lstm_spdin_benchmark_products.ipynb`
- `07_gru_benchmark_products.ipynb`
- `08_tft_benchmark_products.ipynb`
- `09_deepar_benchmark_products.ipynb`
- `10_sarimax_benchmark_products.ipynb`
- `11_hurdle_benchmark_products.ipynb`

Rules used here:

- The source-of-truth values come from metric tables embedded in those notebook outputs.
- CSV artifacts are used only for validation, not as the primary source.
- Notebook 03 / `LSTMForecastModel` is intentionally excluded from these corrected comparison tables because it is not in the requested source-of-truth notebook set.
- Notebook 30 is context only and is not merged into the three 28-day benchmark tables.


## Section A - Benchmark products and context

In [1]:
from pathlib import Path
import json
import math
import re
from html.parser import HTMLParser

import numpy as np
import pandas as pd

ROOT = Path.cwd()
if ROOT.name == "gnn_model_comparison":
    ROOT = ROOT.parents[1]
elif ROOT.name == "notebooks":
    ROOT = ROOT.parent

NOTEBOOK_DIR = ROOT / "notebooks" / "gnn_model_comparison"
REPORT_DIR = ROOT / "reports" / "gnn_benchmarks"
OUTPUT_DIR = ROOT / "reports" / "output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

PRODUCTS = {
    "high_demand": {
        "series_id": "FOODS_3_228_CA_1_validation",
        "benchmark_label": "high_demand_stable",
        "display_name": "High-demand",
        "context": "Highest mean demand in the 16-product panel, almost no zeros, smooth active benchmark.",
    },
    "intermittent": {
        "series_id": "FOODS_2_044_CA_3_validation",
        "benchmark_label": "intermittent",
        "display_name": "Intermittent",
        "context": "Moderate sparsity with recurring demand; chosen as the intermittent benchmark.",
    },
    "low_demand": {
        "series_id": "HOBBIES_1_133_CA_4_validation",
        "benchmark_label": "low_volume",
        "display_name": "Low-demand",
        "context": "Extremely sparse low-volume stress-test product.",
    },
}

MODEL_SOURCES = [
    {
        "model": "LSTM #2",
        "source_model": "WalmartRepoBiLSTM",
        "source_notebook": "notebooks/gnn_model_comparison/04_walmart_repo_lstm_impl2_benchmark_products.ipynb",
        "source_metrics": "reports/gnn_benchmarks/walmart_repo_lstm_impl2/metrics.csv",
    },
    {
        "model": "LSTM #3",
        "source_model": "DeepLearningTimeSeriesRepoLSTM",
        "source_notebook": "notebooks/gnn_model_comparison/05_lstm_deep_learning_time_series_forecasting_benchmark_products.ipynb",
        "source_metrics": "reports/gnn_benchmarks/deep_learning_time_series_repo_lstm_impl3/metrics.csv",
    },
    {
        "model": "LSTM #4",
        "source_model": "SpdinRepoLSTM",
        "source_notebook": "notebooks/gnn_model_comparison/06_lstm_spdin_benchmark_products.ipynb",
        "source_metrics": "reports/gnn_benchmarks/spdin_repo_lstm_impl4/metrics.csv",
    },
    {
        "model": "GRU",
        "source_model": "ZhangxuRepoGRU",
        "source_notebook": "notebooks/gnn_model_comparison/07_gru_benchmark_products.ipynb",
        "source_metrics": "reports/gnn_benchmarks/zhangxu_repo_gru_impl1/metrics.csv",
    },
    {
        "model": "TFT",
        "source_model": "PyTorchForecastingTFT",
        "source_notebook": "notebooks/gnn_model_comparison/08_tft_benchmark_products.ipynb",
        "source_metrics": "reports/gnn_benchmarks/pytorch_forecasting_tft_impl1/metrics.csv",
    },
    {
        "model": "DeepAR",
        "source_model": "PyTorchForecastingDeepAR",
        "source_notebook": "notebooks/gnn_model_comparison/09_deepar_benchmark_products.ipynb",
        "source_metrics": "reports/gnn_benchmarks/pytorch_forecasting_deepar_impl1/metrics.csv",
    },
    {
        "model": "SARIMAX",
        "source_model": "SARIMAX",
        "source_notebook": "notebooks/gnn_model_comparison/10_sarimax_benchmark_products.ipynb",
        "source_metrics": "reports/gnn_benchmarks/sarimax_benchmark_products/metrics.csv",
    },
    {
        "model": "Hurdle",
        "source_model": "HURDLE",
        "source_notebook": "notebooks/gnn_model_comparison/11_hurdle_benchmark_products.ipynb",
        "source_metrics": "reports/gnn_benchmarks/hurdle_benchmark_products/metrics.csv",
    },
]

METRIC_COLUMNS = [
    "mae",
    "rmse",
    "variance_ratio",
    "flat_nonflat",
    "trend_correlation",
    "shape_similarity",
    "peak_detection_rate",
]

NUMERIC_METRIC_COLUMNS = [
    "mae",
    "rmse",
    "variance_ratio",
    "trend_correlation",
    "shape_similarity",
    "peak_detection_rate",
]

REQUIRED_NOTEBOOK_COLUMNS = {
    "series_id",
    "benchmark_label",
    "mae",
    "rmse",
    "variance_ratio",
    "trend_correlation",
    "shape_similarity",
    "peak_detection_rate",
    "flat_nonflat",
}

MODEL_ORDER = [source["model"] for source in MODEL_SOURCES]

pd.DataFrame(PRODUCTS).T.reset_index(names="benchmark_type")


,benchmark_type,series_id,benchmark_label,display_name,context
0,high_demand,FOODS_3_228_CA_1_validation,high_demand_stable,High-demand,"Highest mean demand in the 16-product panel, a..."
1,intermittent,FOODS_2_044_CA_3_validation,intermittent,Intermittent,Moderate sparsity with recurring demand; chose...
2,low_demand,HOBBIES_1_133_CA_4_validation,low_volume,Low-demand,Extremely sparse low-volume stress-test product.


In [2]:
def notebook_exists(rel_path: str) -> bool:
    return (ROOT / rel_path).exists()


notebook_status = pd.DataFrame(
    [
        {
            "notebook": source["source_notebook"],
            "metrics_csv": source["source_metrics"],
            "notebook_found": notebook_exists(source["source_notebook"]),
            "metrics_found": (ROOT / source["source_metrics"]).exists(),
            "model_row": source["model"],
        }
        for source in MODEL_SOURCES
    ]
)

context_source_status = pd.DataFrame(
    [
        {
            "context_source": "notebooks/gnn_model_comparison/12_correlation_matrix_16_products.ipynb",
            "source_role": "product graph / correlation context",
            "found": (ROOT / "notebooks" / "gnn_model_comparison" / "12_correlation_matrix_16_products.ipynb").exists(),
        },
        {
            "context_source": "reports/gnn_benchmarks/correlation_matrix_16_products/final_hybrid_top2_edge_table.csv",
            "source_role": "Notebook 12 generated relationship table",
            "found": (ROOT / "reports" / "gnn_benchmarks" / "correlation_matrix_16_products" / "final_hybrid_top2_edge_table.csv").exists(),
        },
        {
            "context_source": "notebooks/gnn_model_comparison/benchmark_product_selection.md",
            "source_role": "official benchmark product definitions",
            "found": (ROOT / "notebooks" / "gnn_model_comparison" / "benchmark_product_selection.md").exists(),
        },
        {
            "context_source": "notebooks/gnn_model_comparison/common_train_test_protocol.md",
            "source_role": "shared 365-day window and 28-day val/test protocol",
            "found": (ROOT / "notebooks" / "gnn_model_comparison" / "common_train_test_protocol.md").exists(),
        },
    ]
)

notebook_30_path = ROOT / "notebooks" / "30_aurex_lstm_candidate_evaluation.ipynb"
notebook_30_context = {
    "notebook": "notebooks/30_aurex_lstm_candidate_evaluation.ipynb",
    "notebook_found": notebook_30_path.exists(),
    "series_id_in_config": "missing",
    "series_alias_in_config": "missing",
    "test_days_in_config": "missing",
    "context_length_in_config": "missing",
    "output_dir_in_config": "missing",
    "available_output_metrics": "missing",
    "comparison_use": "context only",
}

if notebook_30_path.exists():
    nb30 = json.loads(notebook_30_path.read_text(encoding="utf-8"))
    text30 = "\n".join("".join(cell.get("source", [])) for cell in nb30.get("cells", []))
    for key in ["SERIES_ID", "SERIES_ALIAS", "TEST_DAYS", "CONTEXT_LENGTH", "OUTPUT_DIR"]:
        match = re.search(rf"['\"]{key}['\"]\s*:\s*([^,\n]+)", text30)
        if match:
            notebook_30_context[f"{key.lower()}_in_config"] = match.group(1).strip()

nb30_metric_candidates = [
    ROOT / "reports" / "paper_experiments" / "30_m5_foods_3_228_ca_1_validation" / "metrics.csv",
    ROOT / "reports" / "paper_experiments" / "30_aurex_lstm_candidate" / "metrics.csv",
]
existing_nb30_metrics = [str(path.relative_to(ROOT)) for path in nb30_metric_candidates if path.exists()]
if existing_nb30_metrics:
    notebook_30_context["available_output_metrics"] = "; ".join(existing_nb30_metrics)

display(notebook_status)
display(context_source_status)
display(pd.DataFrame([notebook_30_context]))

,notebook,metrics_csv,notebook_found,metrics_found,model_row
0,notebooks/gnn_model_comparison/04_walmart_repo...,reports/gnn_benchmarks/walmart_repo_lstm_impl2...,True,True,LSTM #2
1,notebooks/gnn_model_comparison/05_lstm_deep_le...,reports/gnn_benchmarks/deep_learning_time_seri...,True,True,LSTM #3
2,notebooks/gnn_model_comparison/06_lstm_spdin_b...,reports/gnn_benchmarks/spdin_repo_lstm_impl4/m...,True,True,LSTM #4
3,notebooks/gnn_model_comparison/07_gru_benchmar...,reports/gnn_benchmarks/zhangxu_repo_gru_impl1/...,True,True,GRU
4,notebooks/gnn_model_comparison/08_tft_benchmar...,reports/gnn_benchmarks/pytorch_forecasting_tft...,True,True,TFT
5,notebooks/gnn_model_comparison/09_deepar_bench...,reports/gnn_benchmarks/pytorch_forecasting_dee...,True,True,DeepAR
6,notebooks/gnn_model_comparison/10_sarimax_benc...,reports/gnn_benchmarks/sarimax_benchmark_produ...,True,True,SARIMAX
7,notebooks/gnn_model_comparison/11_hurdle_bench...,reports/gnn_benchmarks/hurdle_benchmark_produc...,True,True,Hurdle


,context_source,source_role,found
0,notebooks/gnn_model_comparison/12_correlation_...,product graph / correlation context,True
1,reports/gnn_benchmarks/correlation_matrix_16_p...,Notebook 12 generated relationship table,True
2,notebooks/gnn_model_comparison/benchmark_produ...,official benchmark product definitions,True
3,notebooks/gnn_model_comparison/common_train_te...,shared 365-day window and 28-day val/test prot...,True


,notebook,notebook_found,series_id_in_config,series_alias_in_config,test_days_in_config,context_length_in_config,output_dir_in_config,available_output_metrics,comparison_use
0,notebooks/30_aurex_lstm_candidate_evaluation.i...,True,'FOODS_3_228_CA_1_validation','FOODS_3_228_CA_1_val',365,28,ROOT / 'reports' / 'paper_experiments' / '30_m...,reports\paper_experiments\30_aurex_lstm_candid...,context only


Notebook 12 context note: `12_correlation_matrix_16_products.ipynb` and its generated correlation artifacts are treated as product-relationship context, not as forecasting metric sources.

Notebook 30 context note: the notebook config references `FOODS_3_228_CA_1_validation`, the same high-demand product, but it uses a longer `TEST_DAYS=365` setup rather than the shared 28-day benchmark protocol. Therefore, it is useful context for the high-demand product, but the final comparative tables below use the shared benchmark CSV outputs from the model-specific notebooks.

In [3]:
class NotebookHTMLTableParser(HTMLParser):
    def __init__(self):
        super().__init__()
        self.tables = []
        self._in_table = False
        self._in_row = False
        self._in_cell = False
        self._table = []
        self._row = []
        self._cell = []

    def handle_starttag(self, tag, attrs):
        if tag == "table":
            self._in_table = True
            self._table = []
        elif self._in_table and tag == "tr":
            self._in_row = True
            self._row = []
        elif self._in_row and tag in {"th", "td"}:
            self._in_cell = True
            self._cell = []

    def handle_data(self, data):
        if self._in_cell:
            self._cell.append(data)

    def handle_endtag(self, tag):
        if tag in {"th", "td"} and self._in_cell:
            value = "".join(self._cell).strip()
            self._row.append(value)
            self._cell = []
            self._in_cell = False
        elif tag == "tr" and self._in_row:
            if self._row:
                self._table.append(self._row)
            self._row = []
            self._in_row = False
        elif tag == "table" and self._in_table:
            if self._table:
                self.tables.append(self._table)
            self._table = []
            self._in_table = False


def html_tables_to_frames(html: str) -> list[pd.DataFrame]:
    parser = NotebookHTMLTableParser()
    parser.feed(html)
    frames = []
    for table in parser.tables:
        if len(table) < 2:
            continue
        width = max(len(row) for row in table)
        padded = [row + [""] * (width - len(row)) for row in table]
        header = [str(col).strip() for col in padded[0]]
        data = padded[1:]
        df = pd.DataFrame(data, columns=header)
        if df.columns[0] == "" or df.columns[0].startswith("Unnamed"):
            df = df.drop(columns=[df.columns[0]])
        df.columns = [str(col).strip() for col in df.columns]
        frames.append(df)
    return frames


def extract_notebook_metric_table(source: dict) -> tuple[pd.DataFrame, str]:
    path = ROOT / source["source_notebook"]
    if not path.exists():
        return pd.DataFrame(), "missing notebook"

    nb = json.loads(path.read_text(encoding="utf-8"))
    candidates = []
    for cell_index, cell in enumerate(nb.get("cells", [])):
        for output_index, output in enumerate(cell.get("outputs", [])):
            html = output.get("data", {}).get("text/html")
            if html is None:
                continue
            html = "".join(html) if isinstance(html, list) else str(html)
            for frame in html_tables_to_frames(html):
                if REQUIRED_NOTEBOOK_COLUMNS.issubset(set(frame.columns)):
                    score = 1 if "model" in frame.columns else 0
                    candidates.append((score, cell_index, output_index, frame))

    if not candidates:
        return pd.DataFrame(), "no metric table found in notebook outputs"

    # Prefer the latest displayed table that still contains the source model column.
    candidates = sorted(candidates, key=lambda item: (item[0], item[1], item[2]))
    _, cell_index, output_index, frame = candidates[-1]
    out = frame.copy()
    out["_source_cell"] = cell_index
    out["_source_output"] = output_index
    out["_extraction_status"] = "notebook output table"
    return out, f"cell {cell_index}, output {output_index}"


def clean_notebook_metrics(frame: pd.DataFrame, source: dict) -> pd.DataFrame:
    out = frame.copy()
    for col in NUMERIC_METRIC_COLUMNS + ["final_loss", "pred_std", "real_std"]:
        if col in out.columns:
            out[col] = pd.to_numeric(out[col], errors="coerce")
    if "model" not in out.columns:
        out["model"] = source["source_model"]
    return out


def load_csv_metrics(source: dict) -> pd.DataFrame:
    path = ROOT / source["source_metrics"]
    if not path.exists():
        return pd.DataFrame()
    out = pd.read_csv(path)
    for col in NUMERIC_METRIC_COLUMNS + ["final_loss", "pred_std", "real_std"]:
        if col in out.columns:
            out[col] = pd.to_numeric(out[col], errors="coerce")
    return out


def values_match(a, b, tol: float = 1e-9) -> bool:
    if pd.isna(a) and pd.isna(b):
        return True
    if isinstance(a, str) or isinstance(b, str):
        return str(a) == str(b)
    try:
        return abs(float(a) - float(b)) <= tol
    except Exception:
        return str(a) == str(b)


def source_note(row: pd.Series) -> str:
    parts = []
    if pd.notna(row.get("mae")):
        parts.append(f"MAE={row['mae']:.4g}")
    if pd.notna(row.get("variance_ratio")):
        parts.append(f"VR={row['variance_ratio']:.3g}")
    if pd.notna(row.get("flat_nonflat")):
        parts.append(str(row["flat_nonflat"]))
    if pd.notna(row.get("peak_detection_rate")):
        parts.append(f"PDR={row['peak_detection_rate']:.3g}")
    return "; ".join(parts) if parts else "missing metrics in source notebook"


def build_source_notebook_tables() -> tuple[pd.DataFrame, pd.DataFrame]:
    rows = []
    validation_rows = []
    for source in MODEL_SOURCES:
        notebook_df_raw, extraction_location = extract_notebook_metric_table(source)
        notebook_df = clean_notebook_metrics(notebook_df_raw, source) if not notebook_df_raw.empty else pd.DataFrame()
        csv_df = load_csv_metrics(source)

        for benchmark_type, product in PRODUCTS.items():
            base = {
                "benchmark_type": benchmark_type,
                "series_id": product["series_id"],
                "benchmark_label": product["benchmark_label"],
                "model": source["model"],
                "source_model": source["source_model"],
                "source_notebook": source["source_notebook"],
                "source_metrics": source["source_metrics"],
                "notebook_extraction_location": extraction_location,
            }

            if notebook_df.empty or "series_id" not in notebook_df.columns:
                row = {**base, **{col: np.nan for col in METRIC_COLUMNS}}
                row["short_interpretation_note"] = "missing notebook metric table"
                rows.append(row)
                validation_rows.append({**base, "validation_status": "missing notebook metric table"})
                continue

            nb_match = notebook_df[notebook_df["series_id"].astype(str).eq(product["series_id"])]
            if nb_match.empty:
                row = {**base, **{col: np.nan for col in METRIC_COLUMNS}}
                row["short_interpretation_note"] = "missing product row in notebook output"
                rows.append(row)
                validation_rows.append({**base, "validation_status": "missing product row in notebook output"})
                continue

            nb_row = nb_match.iloc[0].to_dict()
            row = base.copy()
            row["benchmark_label"] = nb_row.get("benchmark_label", row["benchmark_label"])
            row["source_model"] = nb_row.get("model", row["source_model"])
            for col in METRIC_COLUMNS:
                row[col] = nb_row.get(col, np.nan)
            row["short_interpretation_note"] = source_note(pd.Series(row))
            rows.append(row)

            csv_status = "csv missing"
            csv_row = {}
            if not csv_df.empty and "series_id" in csv_df.columns:
                csv_match = csv_df[csv_df["series_id"].astype(str).eq(product["series_id"])]
                if not csv_match.empty:
                    csv_row = csv_match.iloc[0].to_dict()
                    mismatches = []
                    for col in METRIC_COLUMNS:
                        nb_val = row.get(col, np.nan)
                        csv_val = csv_row.get(col, np.nan)
                        if not values_match(nb_val, csv_val):
                            mismatches.append(col)
                    csv_status = "matches csv" if not mismatches else "differs from csv: " + ", ".join(mismatches)
                else:
                    csv_status = "csv missing product row"

            validation_entry = {
                **base,
                "notebook_source_model": row["source_model"],
                "notebook_mae": row.get("mae", np.nan),
                "notebook_rmse": row.get("rmse", np.nan),
                "notebook_variance_ratio": row.get("variance_ratio", np.nan),
                "notebook_flat_nonflat": row.get("flat_nonflat", np.nan),
                "notebook_trend_correlation": row.get("trend_correlation", np.nan),
                "notebook_shape_similarity": row.get("shape_similarity", np.nan),
                "notebook_peak_detection_rate": row.get("peak_detection_rate", np.nan),
                "final_mae": row.get("mae", np.nan),
                "final_rmse": row.get("rmse", np.nan),
                "final_variance_ratio": row.get("variance_ratio", np.nan),
                "final_flat_nonflat": row.get("flat_nonflat", np.nan),
                "final_trend_correlation": row.get("trend_correlation", np.nan),
                "final_shape_similarity": row.get("shape_similarity", np.nan),
                "final_peak_detection_rate": row.get("peak_detection_rate", np.nan),
                "csv_validation_status": csv_status,
            }
            validation_rows.append(validation_entry)

    all_tables = pd.DataFrame(rows)
    all_tables["model"] = pd.Categorical(all_tables["model"], categories=MODEL_ORDER, ordered=True)
    all_tables = all_tables.sort_values(["benchmark_type", "model"]).reset_index(drop=True)
    all_tables["model"] = all_tables["model"].astype(str)

    validation_df = pd.DataFrame(validation_rows)
    validation_df["model"] = pd.Categorical(validation_df["model"], categories=MODEL_ORDER, ordered=True)
    validation_df = validation_df.sort_values(["benchmark_type", "model"]).reset_index(drop=True)
    validation_df["model"] = validation_df["model"].astype(str)
    return all_tables, validation_df


all_tables, validation_df = build_source_notebook_tables()

DISPLAY_COLUMNS = [
    "model",
    "source_model",
    "series_id",
    "benchmark_label",
    "mae",
    "rmse",
    "variance_ratio",
    "flat_nonflat",
    "trend_correlation",
    "shape_similarity",
    "peak_detection_rate",
    "short_interpretation_note",
    "source_notebook",
    "notebook_extraction_location",
]


def table_for(benchmark_type: str, save: bool = True) -> pd.DataFrame:
    table = all_tables.loc[all_tables["benchmark_type"].eq(benchmark_type), DISPLAY_COLUMNS].copy()
    table = table.replace({np.nan: "missing"})
    if save:
        filenames = {
            "high_demand": "high_demand_comparison.csv",
            "intermittent": "intermittent_comparison.csv",
            "low_demand": "low_demand_comparison.csv",
        }
        table.to_csv(OUTPUT_DIR / filenames[benchmark_type], index=False)
    return table


saved_tables = {
    "high_demand": table_for("high_demand"),
    "intermittent": table_for("intermittent"),
    "low_demand": table_for("low_demand"),
}

validation_df.to_csv(OUTPUT_DIR / "notebook13_source_validation.csv", index=False)
print(f"Saved corrected tables and validation to: {OUTPUT_DIR}")


Saved corrected tables and validation to: c:\Users\braya\Documents\Research\aurex-demand-forecasting-main\reports\output


## Validation - notebook-extracted values used in final tables

This section verifies the source path for each model. The `notebook_*` columns are extracted from the original notebook output tables. The `final_*` columns are the values written into Sections B-D. CSV artifacts are checked only as validation.


In [4]:
VALIDATION_COLUMNS = [
    "benchmark_type",
    "model",
    "source_notebook",
    "notebook_extraction_location",
    "notebook_source_model",
    "series_id",
    "benchmark_label",
    "notebook_mae",
    "notebook_rmse",
    "notebook_variance_ratio",
    "notebook_flat_nonflat",
    "notebook_trend_correlation",
    "notebook_shape_similarity",
    "notebook_peak_detection_rate",
    "final_mae",
    "final_rmse",
    "final_variance_ratio",
    "final_flat_nonflat",
    "final_trend_correlation",
    "final_shape_similarity",
    "final_peak_detection_rate",
    "csv_validation_status",
]

validation_view = validation_df[VALIDATION_COLUMNS].copy()
validation_view


,benchmark_type,model,source_notebook,notebook_extraction_location,notebook_source_model,series_id,benchmark_label,notebook_mae,notebook_rmse,notebook_variance_ratio,...,notebook_shape_similarity,notebook_peak_detection_rate,final_mae,final_rmse,final_variance_ratio,final_flat_nonflat,final_trend_correlation,final_shape_similarity,final_peak_detection_rate,csv_validation_status
0,high_demand,LSTM #2,notebooks/gnn_model_comparison/04_walmart_repo...,"cell 6, output 4",WalmartRepoBiLSTM,FOODS_3_228_CA_1_validation,high_demand_stable,2.313500e+00,2.835100e+00,0.113300,...,0.716300,0.000,2.313500e+00,2.835100e+00,0.113300,non-flat,0.612100,0.716300,0.000,"differs from csv: mae, rmse, variance_ratio, t..."
1,high_demand,LSTM #3,notebooks/gnn_model_comparison/05_lstm_deep_le...,"cell 5, output 3",DeepLearningTimeSeriesRepoLSTM,FOODS_3_228_CA_1_validation,high_demand_stable,2.291700e+00,2.772600e+00,0.039100,...,0.527600,0.000,2.291700e+00,2.772600e+00,0.039100,flat,0.602200,0.527600,0.000,"differs from csv: mae, rmse, variance_ratio, t..."
2,high_demand,LSTM #4,notebooks/gnn_model_comparison/06_lstm_spdin_b...,"cell 5, output 1",SpdinRepoLSTM,FOODS_3_228_CA_1_validation,high_demand_stable,2.246100e+00,2.767200e+00,0.184400,...,0.678100,0.000,2.246100e+00,2.767200e+00,0.184400,non-flat,0.626500,0.678100,0.000,"differs from csv: mae, rmse, variance_ratio, t..."
3,high_demand,GRU,notebooks/gnn_model_comparison/07_gru_benchmar...,"cell 5, output 0",ZhangxuRepoGRU,FOODS_3_228_CA_1_validation,high_demand_stable,2.669856e+00,3.146730e+00,0.639439,...,0.701566,1.000,2.669856e+00,3.146730e+00,0.639439,non-flat,0.641938,0.701566,1.000,"differs from csv: mae, rmse, variance_ratio, t..."
4,high_demand,TFT,notebooks/gnn_model_comparison/08_tft_benchmar...,"cell 5, output 0",PyTorchForecastingTFT,FOODS_3_228_CA_1_validation,high_demand_stable,2.672164e+00,3.357016e+00,0.731144,...,0.703902,1.000,2.672164e+00,3.357016e+00,0.731144,non-flat,0.527896,0.703902,1.000,"differs from csv: mae, rmse, variance_ratio, t..."
5,high_demand,DeepAR,notebooks/gnn_model_comparison/09_deepar_bench...,"cell 5, output 0",PyTorchForecastingDeepAR,FOODS_3_228_CA_1_validation,high_demand_stable,3.036080e+00,3.645430e+00,0.800634,...,0.718358,1.000,3.036080e+00,3.645430e+00,0.800634,non-flat,0.502705,0.718358,1.000,"differs from csv: mae, rmse, variance_ratio, t..."
6,high_demand,SARIMAX,notebooks/gnn_model_comparison/10_sarimax_benc...,"cell 5, output 0",SARIMAX,FOODS_3_228_CA_1_validation,high_demand_stable,2.836876e+00,3.759589e+00,0.800717,...,0.718502,1.000,2.836876e+00,3.759589e+00,0.800717,non-flat,0.522383,0.718502,1.000,"differs from csv: mae, rmse, variance_ratio, t..."
7,high_demand,Hurdle,notebooks/gnn_model_comparison/11_hurdle_bench...,"cell 5, output 0",HURDLE,FOODS_3_228_CA_1_validation,high_demand_stable,3.239571e+00,4.047823e+00,0.894939,...,0.659871,0.875,3.239571e+00,4.047823e+00,0.894939,non-flat,0.374567,0.659871,0.875,"differs from csv: mae, rmse, variance_ratio, t..."
8,intermittent,LSTM #2,notebooks/gnn_model_comparison/04_walmart_repo...,"cell 6, output 4",WalmartRepoBiLSTM,FOODS_2_044_CA_3_validation,intermittent,1.143900e+00,1.517200e+00,0.022600,...,0.575400,0.000,1.143900e+00,1.517200e+00,0.022600,flat,0.327500,0.575400,0.000,"differs from csv: mae, rmse, variance_ratio, t..."
9,intermittent,LSTM #3,notebooks/gnn_model_comparison/05_lstm_deep_le...,"cell 5, output 3",DeepLearningTimeSeriesRepoLSTM,FOODS_2_044_CA_3_validation,intermittent,1.143400e+00,1.509100e+00,0.009200,...,0.595300,0.000,1.143400e+00,1.509100e+00,0.009200,flat,0.359000,0.595300,0.000,"differs from csv: mae, rmse, variance_ratio, t..."


## Section B - High-demand comparative table

In [5]:
high_demand_table = saved_tables["high_demand"]
high_demand_table

,model,source_model,series_id,benchmark_label,mae,rmse,variance_ratio,flat_nonflat,trend_correlation,shape_similarity,peak_detection_rate,short_interpretation_note,source_notebook,notebook_extraction_location
0,LSTM #2,WalmartRepoBiLSTM,FOODS_3_228_CA_1_validation,high_demand_stable,2.313500,2.835100,0.113300,non-flat,0.612100,0.716300,0.000,MAE=2.313; VR=0.113; non-flat; PDR=0,notebooks/gnn_model_comparison/04_walmart_repo...,"cell 6, output 4"
1,LSTM #3,DeepLearningTimeSeriesRepoLSTM,FOODS_3_228_CA_1_validation,high_demand_stable,2.291700,2.772600,0.039100,flat,0.602200,0.527600,0.000,MAE=2.292; VR=0.0391; flat; PDR=0,notebooks/gnn_model_comparison/05_lstm_deep_le...,"cell 5, output 3"
2,LSTM #4,SpdinRepoLSTM,FOODS_3_228_CA_1_validation,high_demand_stable,2.246100,2.767200,0.184400,non-flat,0.626500,0.678100,0.000,MAE=2.246; VR=0.184; non-flat; PDR=0,notebooks/gnn_model_comparison/06_lstm_spdin_b...,"cell 5, output 1"
3,GRU,ZhangxuRepoGRU,FOODS_3_228_CA_1_validation,high_demand_stable,2.669856,3.146730,0.639439,non-flat,0.641938,0.701566,1.000,MAE=2.67; VR=0.639; non-flat; PDR=1,notebooks/gnn_model_comparison/07_gru_benchmar...,"cell 5, output 0"
4,TFT,PyTorchForecastingTFT,FOODS_3_228_CA_1_validation,high_demand_stable,2.672164,3.357016,0.731144,non-flat,0.527896,0.703902,1.000,MAE=2.672; VR=0.731; non-flat; PDR=1,notebooks/gnn_model_comparison/08_tft_benchmar...,"cell 5, output 0"
5,DeepAR,PyTorchForecastingDeepAR,FOODS_3_228_CA_1_validation,high_demand_stable,3.036080,3.645430,0.800634,non-flat,0.502705,0.718358,1.000,MAE=3.036; VR=0.801; non-flat; PDR=1,notebooks/gnn_model_comparison/09_deepar_bench...,"cell 5, output 0"
6,SARIMAX,SARIMAX,FOODS_3_228_CA_1_validation,high_demand_stable,2.836876,3.759589,0.800717,non-flat,0.522383,0.718502,1.000,MAE=2.837; VR=0.801; non-flat; PDR=1,notebooks/gnn_model_comparison/10_sarimax_benc...,"cell 5, output 0"
7,Hurdle,HURDLE,FOODS_3_228_CA_1_validation,high_demand_stable,3.239571,4.047823,0.894939,non-flat,0.374567,0.659871,0.875,MAE=3.24; VR=0.895; non-flat; PDR=0.875,notebooks/gnn_model_comparison/11_hurdle_bench...,"cell 5, output 0"


## Section C - Intermittent comparative table

In [6]:
intermittent_table = saved_tables["intermittent"]
intermittent_table

,model,source_model,series_id,benchmark_label,mae,rmse,variance_ratio,flat_nonflat,trend_correlation,shape_similarity,peak_detection_rate,short_interpretation_note,source_notebook,notebook_extraction_location
8,LSTM #2,WalmartRepoBiLSTM,FOODS_2_044_CA_3_validation,intermittent,1.143900,1.517200,0.022600,flat,0.327500,0.575400,0.0,MAE=1.144; VR=0.0226; flat; PDR=0,notebooks/gnn_model_comparison/04_walmart_repo...,"cell 6, output 4"
9,LSTM #3,DeepLearningTimeSeriesRepoLSTM,FOODS_2_044_CA_3_validation,intermittent,1.143400,1.509100,0.009200,flat,0.359000,0.595300,0.0,MAE=1.143; VR=0.0092; flat; PDR=0,notebooks/gnn_model_comparison/05_lstm_deep_le...,"cell 5, output 3"
10,LSTM #4,SpdinRepoLSTM,FOODS_2_044_CA_3_validation,intermittent,1.121900,1.502400,0.046500,flat,0.409000,0.519700,0.0,MAE=1.122; VR=0.0465; flat; PDR=0,notebooks/gnn_model_comparison/06_lstm_spdin_b...,"cell 5, output 1"
11,GRU,ZhangxuRepoGRU,FOODS_2_044_CA_3_validation,intermittent,1.143966,1.495846,0.050869,flat,0.350760,0.717551,0.0,MAE=1.144; VR=0.0509; flat; PDR=0,notebooks/gnn_model_comparison/07_gru_benchmar...,"cell 5, output 0"
12,TFT,PyTorchForecastingTFT,FOODS_2_044_CA_3_validation,intermittent,0.997338,1.455829,0.362655,non-flat,0.309375,0.588767,1.0,MAE=0.9973; VR=0.363; non-flat; PDR=1,notebooks/gnn_model_comparison/08_tft_benchmar...,"cell 5, output 0"
13,DeepAR,PyTorchForecastingDeepAR,FOODS_2_044_CA_3_validation,intermittent,1.060929,1.378756,0.191174,non-flat,0.462561,0.607884,0.0,MAE=1.061; VR=0.191; non-flat; PDR=0,notebooks/gnn_model_comparison/09_deepar_bench...,"cell 5, output 0"
14,SARIMAX,SARIMAX,FOODS_2_044_CA_3_validation,intermittent,1.210873,1.587210,0.270567,non-flat,-0.288062,0.674077,0.6,MAE=1.211; VR=0.271; non-flat; PDR=0.6,notebooks/gnn_model_comparison/10_sarimax_benc...,"cell 5, output 0"
15,Hurdle,HURDLE,FOODS_2_044_CA_3_validation,intermittent,1.129554,1.449480,0.137035,non-flat,0.230195,0.685530,0.0,MAE=1.13; VR=0.137; non-flat; PDR=0,notebooks/gnn_model_comparison/11_hurdle_bench...,"cell 5, output 0"


## Section D - Low-demand comparative table

In [7]:
low_demand_table = saved_tables["low_demand"]
low_demand_table

,model,source_model,series_id,benchmark_label,mae,rmse,variance_ratio,flat_nonflat,trend_correlation,shape_similarity,peak_detection_rate,short_interpretation_note,source_notebook,notebook_extraction_location
16,LSTM #2,WalmartRepoBiLSTM,HOBBIES_1_133_CA_4_validation,low_volume,1.390000e-02,1.410000e-02,0.0,flat,missing,0.519900,1.0,MAE=0.0139; VR=0; flat; PDR=1,notebooks/gnn_model_comparison/04_walmart_repo...,"cell 6, output 4"
17,LSTM #3,DeepLearningTimeSeriesRepoLSTM,HOBBIES_1_133_CA_4_validation,low_volume,1.690000e-02,1.720000e-02,0.0,flat,missing,0.494200,1.0,MAE=0.0169; VR=0; flat; PDR=1,notebooks/gnn_model_comparison/05_lstm_deep_le...,"cell 5, output 3"
18,LSTM #4,SpdinRepoLSTM,HOBBIES_1_133_CA_4_validation,low_volume,3.520000e-02,3.530000e-02,0.0,flat,missing,0.177200,1.0,MAE=0.0352; VR=0; flat; PDR=1,notebooks/gnn_model_comparison/06_lstm_spdin_b...,"cell 5, output 1"
19,GRU,ZhangxuRepoGRU,HOBBIES_1_133_CA_4_validation,low_volume,3.680900e-02,3.708800e-02,0.0,flat,missing,0.425688,1.0,MAE=0.03681; VR=0; flat; PDR=1,notebooks/gnn_model_comparison/07_gru_benchmar...,"cell 5, output 0"
20,TFT,PyTorchForecastingTFT,HOBBIES_1_133_CA_4_validation,low_volume,2.404973e-09,6.301541e-09,0.0,flat,missing,0.917428,1.0,MAE=2.405e-09; VR=0; flat; PDR=1,notebooks/gnn_model_comparison/08_tft_benchmar...,"cell 5, output 0"
21,DeepAR,PyTorchForecastingDeepAR,HOBBIES_1_133_CA_4_validation,low_volume,1.760200e-02,1.760900e-02,0.0,flat,missing,0.380101,1.0,MAE=0.0176; VR=0; flat; PDR=1,notebooks/gnn_model_comparison/09_deepar_bench...,"cell 5, output 0"
22,SARIMAX,SARIMAX,HOBBIES_1_133_CA_4_validation,low_volume,0.000000e+00,0.000000e+00,0.0,flat,missing,1.000000,1.0,MAE=0; VR=0; flat; PDR=1,notebooks/gnn_model_comparison/10_sarimax_benc...,"cell 5, output 0"
23,Hurdle,HURDLE,HOBBIES_1_133_CA_4_validation,low_volume,4.546000e-03,5.136000e-03,0.0,flat,missing,0.688347,1.0,MAE=0.004546; VR=0; flat; PDR=1,notebooks/gnn_model_comparison/11_hurdle_bench...,"cell 5, output 0"


## Section E - Ranking and takeaway

In [8]:
BASELINE_MODELS = {"SARIMAX", "Hurdle"}


def numeric_frame(benchmark_type: str) -> pd.DataFrame:
    table = all_tables[all_tables["benchmark_type"].eq(benchmark_type)].copy()
    for col in ["mae", "rmse", "variance_ratio", "trend_correlation", "shape_similarity", "peak_detection_rate"]:
        table[col] = pd.to_numeric(table[col], errors="coerce")
    return table


def rank_benchmark(benchmark_type: str) -> dict:
    table = numeric_frame(benchmark_type)
    available = table.dropna(subset=["mae"])
    if available.empty:
        return {
            "benchmark_type": benchmark_type,
            "strongest_model": "missing",
            "strongest_baseline": "missing",
            "flat_nonflat": "missing",
            "explanation": "No MAE values available.",
        }

    winner = available.sort_values(["mae", "rmse"], ascending=[True, True]).iloc[0]
    baselines = available[available["model"].isin(BASELINE_MODELS)]
    baseline = baselines.sort_values(["mae", "rmse"], ascending=[True, True]).iloc[0] if not baselines.empty else None

    explanation = (
        f"{winner['model']} ranks first by lowest MAE ({winner['mae']:.4g}) and RMSE ({winner['rmse']:.4g}). "
        f"Its prediction is {winner['flat_nonflat']}."
    )

    if benchmark_type == "low_demand":
        explanation += " The low-demand test window has zero real standard deviation in these outputs, so flat all-zero behavior can score perfectly without proving useful demand learning."

    return {
        "benchmark_type": benchmark_type,
        "benchmark_display": PRODUCTS[benchmark_type]["display_name"],
        "strongest_model": winner["model"],
        "strongest_model_mae": winner["mae"],
        "strongest_model_flat_nonflat": winner["flat_nonflat"],
        "strongest_baseline": baseline["model"] if baseline is not None else "missing",
        "strongest_baseline_mae": baseline["mae"] if baseline is not None else np.nan,
        "strongest_baseline_flat_nonflat": baseline["flat_nonflat"] if baseline is not None else "missing",
        "explanation": explanation,
    }


ranking_df = pd.DataFrame([rank_benchmark(key) for key in PRODUCTS])
ranking_df

,benchmark_type,benchmark_display,strongest_model,strongest_model_mae,strongest_model_flat_nonflat,strongest_baseline,strongest_baseline_mae,strongest_baseline_flat_nonflat,explanation
0,high_demand,High-demand,LSTM #4,2.246100,non-flat,SARIMAX,2.836876,non-flat,LSTM #4 ranks first by lowest MAE (2.246) and ...
1,intermittent,Intermittent,TFT,0.997338,non-flat,Hurdle,1.129554,non-flat,TFT ranks first by lowest MAE (0.9973) and RMS...
2,low_demand,Low-demand,SARIMAX,0.000000,flat,SARIMAX,0.000000,flat,SARIMAX ranks first by lowest MAE (0) and RMSE...


## Section F - Final summary

In [9]:
summary_lines = []
for _, row in ranking_df.iterrows():
    summary_lines.append(
        f"{row['benchmark_display']}: strongest model is {row['strongest_model']} "
        f"(MAE={row['strongest_model_mae']:.4g}, {row['strongest_model_flat_nonflat']}); "
        f"strongest baseline is {row['strongest_baseline']} "
        f"(MAE={row['strongest_baseline_mae']:.4g}, {row['strongest_baseline_flat_nonflat']})."
    )

summary_lines.append(
    "Low-demand usefulness caution: in the source metrics, real_std is 0.0 for the low-volume test window; "
    "therefore zero or near-zero forecasts can look excellent while offering limited practical signal."
)
summary_lines.append(
    "Notebook 30 fits as context for FOODS_3_228_CA_1_validation and the LSTM candidate setup, "
    "but its 365-day test setup is not merged into the 28-day shared benchmark tables."
)

print("\n".join(f"- {line}" for line in summary_lines))

print("\nCSV outputs:")
for filename in ["high_demand_comparison.csv", "intermittent_comparison.csv", "low_demand_comparison.csv"]:
    print(f"- {OUTPUT_DIR / filename}")

- High-demand: strongest model is LSTM #4 (MAE=2.246, non-flat); strongest baseline is SARIMAX (MAE=2.837, non-flat).
- Intermittent: strongest model is TFT (MAE=0.9973, non-flat); strongest baseline is Hurdle (MAE=1.13, non-flat).
- Low-demand: strongest model is SARIMAX (MAE=0, flat); strongest baseline is SARIMAX (MAE=0, flat).
- Low-demand usefulness caution: in the source metrics, real_std is 0.0 for the low-volume test window; therefore zero or near-zero forecasts can look excellent while offering limited practical signal.
- Notebook 30 fits as context for FOODS_3_228_CA_1_validation and the LSTM candidate setup, but its 365-day test setup is not merged into the 28-day shared benchmark tables.

CSV outputs:
- c:\Users\braya\Documents\Research\aurex-demand-forecasting-main\reports\output\high_demand_comparison.csv
- c:\Users\braya\Documents\Research\aurex-demand-forecasting-main\reports\output\intermittent_comparison.csv
- c:\Users\braya\Documents\Research\aurex-demand-forecasting